# AutoRouteAI — Sentiment Analysis Demo

Demonstrates how driver feedback is analysed to score roads.

**Steps:**
1. Load feedback samples
2. Run sentiment classification
3. Extract road aspects
4. Aggregate per-road sentiment scores


In [ ]:
import sys
sys.path.insert(0, '../backend')

from ai.sentiment_model import SentimentAnalyser
import pandas as pd

analyser = SentimentAnalyser()
print('Analyser ready')

In [ ]:
# Sample driver feedback
samples = [
    {'road': 'Anna Salai',       'text': 'Very smooth road, great surface. Fast in the morning.'},
    {'road': 'Anna Salai',       'text': 'Traffic is horrible near Gemini flyover every evening.'},
    {'road': 'T. Nagar stretch', 'text': 'Full of potholes near Pondy Bazaar. Very bad road quality.'},
    {'road': 'T. Nagar stretch', 'text': 'Narrow but manageable if you know the shortcuts.'},
    {'road': 'OMR',              'text': 'Best road in Chennai. Super smooth asphalt.'},
    {'road': 'OMR',              'text': 'Floods near Sholinganallur during monsoon. Dangerous.'},
    {'road': 'ECR',              'text': 'Good road, wide lanes, not much traffic after 9 AM.'},
    {'road': 'Mount Road',       'text': 'Romba nalla road in the morning. Super fast.'},
    {'road': 'Mount Road',       'text': 'Ketta road at peak hours. Romba mosam traffic.'},
]

results = []
for s in samples:
    lang = 'ta' if any(w in s['text'] for w in ['romba', 'nalla', 'ketta', 'mosam']) else 'en'
    r = analyser.analyse(s['text'], language=lang)
    results.append({
        'road':    s['road'],
        'text':    s['text'][:60] + '...',
        'label':   r['label'],
        'score':   r['score'],
        'aspects': ', '.join(f"{k}={v}" for k, v in r['road_aspects'].items()),
    })

df = pd.DataFrame(results)
df

In [ ]:
# Aggregate sentiment per road
def label_to_score(label):
    return {'positive': 1.0, 'neutral': 0.5, 'negative': 0.0}.get(label, 0.5)

df['numeric_score'] = df['label'].apply(label_to_score)
road_scores = df.groupby('road')['numeric_score'].mean().sort_values(ascending=False)

print('Road Sentiment Scores (higher = more positive):')
print(road_scores.to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#22c55e' if s > 0.6 else '#f59e0b' if s > 0.4 else '#ef4444' for s in road_scores.values]
road_scores.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Sentiment Score (0=negative, 1=positive)')
ax.set_title('AutoRouteAI — Road Sentiment from Driver Feedback')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.savefig('../docs/sentiment_scores.png', dpi=150)
plt.show()